In [3]:
import pandas as pd
import numpy as np
import time
import os

print("Inizializzazione Motore di Backtest - Setup 1: Assorbimento...")
start_time = time.time()

# --- 1. CARICAMENTO DATI (Smart Cross-Platform Path) ---
home_dir = os.path.expanduser('~')

# Lista dei percorsi possibili (adattabile a Mac e Windows con il tuo setup GitHub/VSCode)
percorsi_possibili = [
    os.path.join(home_dir, "source", "repos", "QuantEdge_Project", "data", "processed", "es_master_4months.parquet"),
    os.path.join(os.getcwd(), "data", "processed", "es_master_4months.parquet"),
    os.path.join(os.getcwd(), "..", "data", "processed", "es_master_4months.parquet")
]

# Cerchiamo il file scorrendo la lista
percorso_parquet = None
for p in percorsi_possibili:
    if os.path.exists(p):
        percorso_parquet = p
        break

# Se non lo trova, si ferma
if percorso_parquet is None:
    print("\n" + "="*60)
    print("ERRORE CRITICO: File Parquet non trovato in nessuno dei percorsi noti!")
    print("Assicurati di aver clonato la repo e scaricato il file dati su questo PC.")
    print("="*60 + "\n")
    raise FileNotFoundError("File dei dati mancante.")

print(f"Caricamento dataset da:\n-> {percorso_parquet}")

# Carichiamo i dati eliminando immediatamente i NaN finali
df = pd.read_parquet(percorso_parquet).dropna(subset=['bid', 'ask'])

# Impostiamo il tempo come indice e lo ordiniamo (fondamentale per le operazioni Rolling)
df = df.set_index('Datetime_UTC')
df = df.sort_index()
print(f"Dati caricati. Righe operative: {len(df):,}")

# --- 2. PARAMETRI DELLA STRATEGIA (Da Calibrare) ---
FINESTRA_MS = '5S'           # Finestra di osservazione (5 secondi)
SOGLIA_VOLUME = 500          # Volume totale minimo scambiato nei 5 secondi
SOGLIA_DELTA = 300           # Delta minimo (sbilanciamento) nei 5 secondi
MAX_TICK_MOUVEMENT = 1.00    # Il prezzo non deve muoversi più di 1 punto intero (4 tick)

# --- 3. COSTRUZIONE DEI SEGNALI (VETTORIALE) ---
print("Calcolo aggregazioni volumetriche (Rolling Window)...")

# Calcoliamo il volume cumulativo e il delta cumulativo ogni 5 secondi
df['Rolling_Volume'] = df['Volume'].rolling(window=FINESTRA_MS).sum()
df['Rolling_Delta'] = df['Delta'].rolling(window=FINESTRA_MS).sum()

# Calcoliamo il movimento massimo del prezzo negli ultimi 5 secondi
df['Rolling_Max_Price'] = df['Price'].rolling(window=FINESTRA_MS).max()
df['Rolling_Min_Price'] = df['Price'].rolling(window=FINESTRA_MS).min()
df['Price_Range'] = df['Rolling_Max_Price'] - df['Rolling_Min_Price']

print("Ricerca dei Trigger di Assorbimento...")

# TRIGGER LONG: Troviamo quando i Venditori Aggressivi vengono Assorbiti
df['Signal_Long'] = np.where(
    (df['Rolling_Delta'] <= -SOGLIA_DELTA) & 
    (df['Rolling_Volume'] >= SOGLIA_VOLUME) & 
    (df['Price_Range'] <= MAX_TICK_MOUVEMENT),
    1, 0
)

# TRIGGER SHORT: Troviamo quando i Compratori Aggressivi vengono Assorbiti
df['Signal_Short'] = np.where(
    (df['Rolling_Delta'] >= SOGLIA_DELTA) & 
    (df['Rolling_Volume'] >= SOGLIA_VOLUME) & 
    (df['Price_Range'] <= MAX_TICK_MOUVEMENT),
    -1, 0
)

# Uniamo i segnali in una singola colonna (-1 per Short, 0 per Nessun Segnale, 1 per Long)
df['Signal'] = df['Signal_Long'] + df['Signal_Short']

# Isoliamo solo gli istanti esatti in cui scatta il segnale
trade_signals = df[df['Signal'] != 0].copy()

# ---> PONTE FONDAMENTALE <---
# Creiamo la lista valid_trades che verrà utilizzata dallo Script 2 (Il Motore di Backtest)
valid_trades = trade_signals.reset_index().to_dict('records')

print(f"\n--- SCANSIONE COMPLETATA ---")
print(f"Trovati {len(trade_signals)} potenziali setup di Assorbimento Istituzionale.")

# Pulizia memoria (eliminiamo dal dataframe originale le colonne pesanti che non servono più)
df = df.drop(columns=['Rolling_Volume', 'Rolling_Delta', 'Rolling_Max_Price', 'Rolling_Min_Price', 'Price_Range', 'Signal_Long', 'Signal_Short'])

end_time = time.time()
print(f"Tempo totale di calcolo: {(end_time - start_time):.2f} secondi\n")

# --- 4. OUTPUT UNIVERSALE ---
# Mostriamo le prime 10 anomalie trovate in modo compatibile con tutti gli ambienti
try:
    from IPython.display import display
    display(trade_signals[['Price', 'Rolling_Volume', 'Rolling_Delta', 'Price_Range', 'Signal', 'bid', 'ask']].head(10))
except ImportError:
    print(trade_signals[['Price', 'Rolling_Volume', 'Rolling_Delta', 'Price_Range', 'Signal', 'bid', 'ask']].head(10).to_string())

Inizializzazione Motore di Backtest - Setup 1: Assorbimento...
Caricamento dataset da:
-> C:\Users\Leona\source\repos\QuantEdge_Project\data\processed\es_master_4months.parquet
Dati caricati. Righe operative: 71,651,488
Calcolo aggregazioni volumetriche (Rolling Window)...


C:\Users\Leona\AppData\Local\Temp\ipykernel_21052\2567069208.py:54: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Volume'] = df['Volume'].rolling(window=FINESTRA_MS).sum()
C:\Users\Leona\AppData\Local\Temp\ipykernel_21052\2567069208.py:55: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Delta'] = df['Delta'].rolling(window=FINESTRA_MS).sum()
C:\Users\Leona\AppData\Local\Temp\ipykernel_21052\2567069208.py:58: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Max_Price'] = df['Price'].rolling(window=FINESTRA_MS).max()
C:\Users\Leona\AppData\Local\Temp\ipykernel_21052\2567069208.py:59: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Min_Price'] = df['Price'].rolling(window=FINESTRA_MS).min()


Ricerca dei Trigger di Assorbimento...

--- SCANSIONE COMPLETATA ---
Trovati 104924 potenziali setup di Assorbimento Istituzionale.
Tempo totale di calcolo: 35.59 secondi



,Price,Rolling_Volume,Rolling_Delta,Price_Range,Signal,bid,ask
Datetime_UTC,,,,,,,
2026-02-09 13:15:55,6952.75,502.0,-416.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,503.0,-415.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,504.0,-414.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,505.0,-413.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,507.0,-415.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,508.0,-416.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,509.0,-417.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,510.0,-418.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55,6952.75,512.0,-420.0,0.75,1,6917.67,6917.92


In [4]:
import pandas as pd
import numpy as np
import time

print("Avvio Motore di Esecuzione e Risk Management...")
start_sim = time.time()

# --- 0. CONTROLLO DI SICUREZZA ---
# Verifichiamo che i dati generati dallo Script 1 siano presenti in memoria
if 'trade_signals' not in locals() or 'df' not in locals():
    print("\n" + "="*60)
    print("ERRORE: Dati mancanti in memoria!")
    print("Assicurati di aver eseguito lo 'Script 1' (Generatore di Segnali)")
    print("prima di lanciare questo motore di esecuzione.")
    print("="*60 + "\n")
    raise NameError("Variabili 'trade_signals' o 'df' non definite.")

# --- 1. APPLICAZIONE DEL COOLDOWN (Filtro dei Cluster) ---
COOLDOWN_SEC = 10  # Secondi di pausa obbligatoria dopo un trade

# Resettiamo l'indice per calcolare la differenza di tempo
signals_df = trade_signals.reset_index()

valid_trades = []
last_trade_time = None  # Usiamo None al posto di date fittizie per evitare Overflow

for idx, row in signals_df.iterrows():
    # Se è il primo trade in assoluto, o se è passato il tempo di Cooldown
    if last_trade_time is None or (row['Datetime_UTC'] - last_trade_time).total_seconds() >= COOLDOWN_SEC:
        valid_trades.append(row)
        last_trade_time = row['Datetime_UTC']

df_trades = pd.DataFrame(valid_trades)
print(f"Segnali Grezzi: {len(trade_signals):,} -> Trade Effettivi (dopo Cooldown): {len(df_trades):,}")

# --- 2. RISK ENGINE (SIMULAZIONE SL, TP E TIME STOP) ---
print("Simulazione esecuzione trades sul CFD (Applicazione Spread, TP, SL)...")

# Parametri operativi
TP_PUNTI = 1.50
SL_PUNTI = 1.00
TIME_STOP_SEC = 10

# Liste per salvare i risultati
esiti = []
pnl_list = []

for _, trade in df_trades.iterrows():
    t_ingresso = trade['Datetime_UTC']
    t_scadenza = t_ingresso + pd.Timedelta(seconds=TIME_STOP_SEC)
    direzione = trade['Signal']
    
    # Isola la fetta di mercato (i 10 secondi successivi al segnale)
    finestra_futura = df.loc[t_ingresso : t_scadenza]
    
    if finestra_futura.empty:
        continue
        
    # LOGICA LONG
    if direzione == 1:
        prezzo_ingresso = trade['ask'] # Compriamo sull'Ask (più alto)
        target_price = prezzo_ingresso + TP_PUNTI
        stop_price = prezzo_ingresso - SL_PUNTI
        
        hit_tp = (finestra_futura['bid'] >= target_price).idxmax() if (finestra_futura['bid'] >= target_price).any() else None
        hit_sl = (finestra_futura['bid'] <= stop_price).idxmax() if (finestra_futura['bid'] <= stop_price).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl:
                esiti.append('WIN')
                pnl_list.append(TP_PUNTI)
            else:
                esiti.append('LOSS')
                pnl_list.append(-SL_PUNTI)
        elif hit_tp:
            esiti.append('WIN')
            pnl_list.append(TP_PUNTI)
        elif hit_sl:
            esiti.append('LOSS')
            pnl_list.append(-SL_PUNTI)
        else:
            prezzo_uscita = finestra_futura['bid'].iloc[-1]
            esiti.append('TIME_STOP')
            pnl_list.append(prezzo_uscita - prezzo_ingresso)
            
    # LOGICA SHORT
    elif direzione == -1:
        prezzo_ingresso = trade['bid'] # Vendiamo sul Bid (più basso)
        target_price = prezzo_ingresso - TP_PUNTI
        stop_price = prezzo_ingresso + SL_PUNTI
        
        hit_tp = (finestra_futura['ask'] <= target_price).idxmax() if (finestra_futura['ask'] <= target_price).any() else None
        hit_sl = (finestra_futura['ask'] >= stop_price).idxmax() if (finestra_futura['ask'] >= stop_price).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl:
                esiti.append('WIN')
                pnl_list.append(TP_PUNTI)
            else:
                esiti.append('LOSS')
                pnl_list.append(-SL_PUNTI)
        elif hit_tp:
            esiti.append('WIN')
            pnl_list.append(TP_PUNTI)
        elif hit_sl:
            esiti.append('LOSS')
            pnl_list.append(-SL_PUNTI)
        else:
            prezzo_uscita = finestra_futura['ask'].iloc[-1]
            esiti.append('TIME_STOP')
            pnl_list.append(prezzo_ingresso - prezzo_uscita)

df_trades['Esito'] = esiti
df_trades['PnL_Punti'] = pnl_list

# --- 3. REPORT FINALE ---
win_rate = (len(df_trades[df_trades['Esito'] == 'WIN']) / len(df_trades)) * 100 if len(df_trades) > 0 else 0
tot_pnl = sum(pnl_list)
gross_profit = tot_pnl * 50 # 1 punto ES = 50 dollari

print(f"\n--- REPORT BACKTEST (4 MESI) ---")
print(f"Trade Eseguiti (dopo Cooldown): {len(df_trades)}")
print(f"Vittorie (TP Hit): {len(df_trades[df_trades['Esito'] == 'WIN'])}")
print(f"Sconfitte (SL Hit): {len(df_trades[df_trades['Esito'] == 'LOSS'])}")
print(f"Uscite per Tempo (Time Stop): {len(df_trades[df_trades['Esito'] == 'TIME_STOP'])}")
print(f"Win Rate Reale: {win_rate:.2f}%")
print(f"Net PnL (in punti indice): {tot_pnl:.2f} Punti")
print(f"Profitto Stimato (1 Lotto ES): ${gross_profit:,.2f}")
print(f"Tempo di simulazione: {(time.time() - start_sim):.2f} sec")

Avvio Motore di Esecuzione e Risk Management...
Segnali Grezzi: 104,924 -> Trade Effettivi (dopo Cooldown): 766
Simulazione esecuzione trades sul CFD (Applicazione Spread, TP, SL)...

--- REPORT BACKTEST (4 MESI) ---
Trade Eseguiti (dopo Cooldown): 766
Vittorie (TP Hit): 29
Sconfitte (SL Hit): 194
Uscite per Tempo (Time Stop): 543
Win Rate Reale: 3.79%
Net PnL (in punti indice): -177.88 Punti
Profitto Stimato (1 Lotto ES): $-8,894.00
Tempo di simulazione: 6.14 sec


In [5]:
import pandas as pd
import numpy as np
import time

print("Avvio Ottimizzazione a Griglia (Grid Search) sui Risk Parameters...")
start_opt = time.time()

# --- 0. CONTROLLO DI SICUREZZA ---
# Verifichiamo che i dati generati dallo Script 1 siano presenti in memoria
if 'valid_trades' not in locals() or 'df' not in locals():
    print("\n" + "="*60)
    print("ERRORE: Dati mancanti in memoria!")
    print("Assicurati di aver eseguito lo 'Script 1' (Generatore di Segnali)")
    print("prima di lanciare questo motore di ottimizzazione.")
    print("="*60 + "\n")
    raise NameError("Variabili 'valid_trades' o 'df' non definite.")

# --- 1. SETUP DEI PARAMETRI ---
# Parametri da testare (in punti indice)
# Proveremo TP cortissimi (per fare vero scalping) e SL variabili
tp_list = [0.50, 0.75, 1.00, 1.25, 1.50]
sl_list = [0.50, 0.75, 1.00, 1.25, 1.50]
TIME_STOP_SEC = 10

risultati_ottimizzazione = []

# --- 2. MOTORE DI ITERAZIONE ---
# Iteriamo su tutte le combinazioni possibili di TP e SL
for TP in tp_list:
    for SL in sl_list:
        pnl_totale = 0
        win_count = 0
        loss_count = 0
        time_stop_count = 0
        
        # Testiamo la combinazione sui trades che abbiamo già filtrato
        for trade in valid_trades:
            t_ingresso = trade['Datetime_UTC']
            t_scadenza = t_ingresso + pd.Timedelta(seconds=TIME_STOP_SEC)
            direzione = trade['Signal']
            
            # Isoliamo i 10 secondi successivi
            finestra_futura = df.loc[t_ingresso : t_scadenza]
            if finestra_futura.empty:
                continue
                
            if direzione == 1: # LONG
                prezzo_ingresso = trade['ask']
                target_price = prezzo_ingresso + TP
                stop_price = prezzo_ingresso - SL
                
                hit_tp = (finestra_futura['bid'] >= target_price).idxmax() if (finestra_futura['bid'] >= target_price).any() else None
                hit_sl = (finestra_futura['bid'] <= stop_price).idxmax() if (finestra_futura['bid'] <= stop_price).any() else None
                
                if hit_tp and hit_sl:
                    if hit_tp < hit_sl:
                        pnl_totale += TP
                        win_count += 1
                    else:
                        pnl_totale -= SL
                        loss_count += 1
                elif hit_tp:
                    pnl_totale += TP
                    win_count += 1
                elif hit_sl:
                    pnl_totale -= SL
                    loss_count += 1
                else:
                    prezzo_uscita = finestra_futura['bid'].iloc[-1]
                    pnl_totale += (prezzo_uscita - prezzo_ingresso)
                    time_stop_count += 1
                    
            elif direzione == -1: # SHORT
                prezzo_ingresso = trade['bid']
                target_price = prezzo_ingresso - TP
                stop_price = prezzo_ingresso + SL
                
                hit_tp = (finestra_futura['ask'] <= target_price).idxmax() if (finestra_futura['ask'] <= target_price).any() else None
                hit_sl = (finestra_futura['ask'] >= stop_price).idxmax() if (finestra_futura['ask'] >= stop_price).any() else None
                
                if hit_tp and hit_sl:
                    if hit_tp < hit_sl:
                        pnl_totale += TP
                        win_count += 1
                    else:
                        pnl_totale -= SL
                        loss_count += 1
                elif hit_tp:
                    pnl_totale += TP
                    win_count += 1
                elif hit_sl:
                    pnl_totale -= SL
                    loss_count += 1
                else:
                    prezzo_uscita = finestra_futura['ask'].iloc[-1]
                    pnl_totale += (prezzo_ingresso - prezzo_uscita)
                    time_stop_count += 1
        
        # Salviamo il risultato di questa combinazione
        win_rate = (win_count / len(valid_trades)) * 100 if len(valid_trades) > 0 else 0
        risultati_ottimizzazione.append({
            'Take_Profit': TP,
            'Stop_Loss': SL,
            'Net_PnL': round(pnl_totale, 2),
            'Win_Rate_%': round(win_rate, 2),
            'Wins': win_count,
            'Losses': loss_count,
            'Time_Stops': time_stop_count
        })

# --- 3. OUTPUT RISULTATI ---
# Creiamo un DataFrame con i risultati e lo ordiniamo per profitto decrescente
df_results = pd.DataFrame(risultati_ottimizzazione)
df_results = df_results.sort_values(by='Net_PnL', ascending=False).reset_index(drop=True)

print(f"\nOttimizzazione completata in {(time.time() - start_opt):.2f} secondi.")
print("Migliori 10 Configurazioni di Rischio:")

# Gestione cross-platform dell'output tabellare
try:
    from IPython.display import display
    display(df_results.head(10))
except ImportError:
    print(df_results.head(10).to_string())

Avvio Ottimizzazione a Griglia (Grid Search) sui Risk Parameters...

Ottimizzazione completata in 5.29 secondi.
Migliori 10 Configurazioni di Rischio:


,Take_Profit,Stop_Loss,Net_PnL,Win_Rate_%,Wins,Losses,Time_Stops
0,1.50,0.50,-152.47,3.00,23,476,267
1,1.25,0.50,-154.97,4.05,31,476,259
2,0.50,0.50,-156.47,20.89,160,463,143
3,0.75,0.50,-156.84,11.75,90,473,203
4,1.00,0.50,-158.46,6.14,47,475,244
5,1.50,0.75,-170.58,3.66,28,320,418
6,0.50,0.75,-173.03,24.93,191,308,267
7,1.25,0.75,-174.08,4.96,38,320,408
8,0.75,0.75,-174.85,14.10,108,317,341
9,1.50,1.00,-177.88,3.79,29,194,543


In [6]:
import pandas as pd
import numpy as np
import time

print("Avvio Backtest su FUTURE puro (Isolamento dell'Alpha)...")
start_opt = time.time()

# --- 0. CONTROLLO DI SICUREZZA ---
if 'valid_trades' not in locals() or 'df' not in locals():
    print("\n" + "="*60)
    print("ERRORE: Dati mancanti in memoria!")
    print("Assicurati di aver eseguito lo 'Script 1' (Generatore di Segnali)")
    print("prima di lanciare questo motore di backtest.")
    print("="*60 + "\n")
    raise NameError("Variabili 'valid_trades' o 'df' non definite.")

# --- 1. SETUP DEI PARAMETRI ---
# Parametri da testare
tp_list = [0.50, 0.75, 1.00, 1.25, 1.50]
sl_list = [0.50, 0.75, 1.00, 1.25, 1.50]
TIME_STOP_SEC = 10

# Commissioni Futures (Circa 5$ a trade = 0.10 punti indice)
COMMISSIONI_PT = 0.10 

risultati_future = []

# --- 2. MOTORE DI ITERAZIONE ---
# Iteriamo sulle combinazioni
for TP in tp_list:
    for SL in sl_list:
        pnl_totale = 0
        win_count = 0
        loss_count = 0
        time_stop_count = 0
        
        for trade in valid_trades:
            t_ingresso = trade['Datetime_UTC']
            t_scadenza = t_ingresso + pd.Timedelta(seconds=TIME_STOP_SEC)
            direzione = trade['Signal']
            
            # Isoliamo i 10 secondi successivi
            finestra_futura = df.loc[t_ingresso : t_scadenza]
            if finestra_futura.empty:
                continue
                
            # IMPORTANTE: Entriamo al prezzo esatto battuto dal Future, niente spread CFD
            prezzo_ingresso = trade['Price']
                
            if direzione == 1: # LONG
                target_price = prezzo_ingresso + TP
                stop_price = prezzo_ingresso - SL
                
                hit_tp = (finestra_futura['Price'] >= target_price).idxmax() if (finestra_futura['Price'] >= target_price).any() else None
                hit_sl = (finestra_futura['Price'] <= stop_price).idxmax() if (finestra_futura['Price'] <= stop_price).any() else None
                
                if hit_tp and hit_sl:
                    if hit_tp < hit_sl:
                        pnl_totale += (TP - COMMISSIONI_PT)
                        win_count += 1
                    else:
                        pnl_totale -= (SL + COMMISSIONI_PT)
                        loss_count += 1
                elif hit_tp:
                    pnl_totale += (TP - COMMISSIONI_PT)
                    win_count += 1
                elif hit_sl:
                    pnl_totale -= (SL + COMMISSIONI_PT)
                    loss_count += 1
                else:
                    prezzo_uscita = finestra_futura['Price'].iloc[-1]
                    pnl_totale += (prezzo_uscita - prezzo_ingresso) - COMMISSIONI_PT
                    time_stop_count += 1
                    
            elif direzione == -1: # SHORT
                target_price = prezzo_ingresso - TP
                stop_price = prezzo_ingresso + SL
                
                hit_tp = (finestra_futura['Price'] <= target_price).idxmax() if (finestra_futura['Price'] <= target_price).any() else None
                hit_sl = (finestra_futura['Price'] >= stop_price).idxmax() if (finestra_futura['Price'] >= stop_price).any() else None
                
                if hit_tp and hit_sl:
                    if hit_tp < hit_sl:
                        pnl_totale += (TP - COMMISSIONI_PT)
                        win_count += 1
                    else:
                        pnl_totale -= (SL + COMMISSIONI_PT)
                        loss_count += 1
                elif hit_tp:
                    pnl_totale += (TP - COMMISSIONI_PT)
                    win_count += 1
                elif hit_sl:
                    pnl_totale -= (SL + COMMISSIONI_PT)
                    loss_count += 1
                else:
                    prezzo_uscita = finestra_futura['Price'].iloc[-1]
                    pnl_totale += (prezzo_ingresso - prezzo_uscita) - COMMISSIONI_PT
                    time_stop_count += 1
        
        win_rate = (win_count / len(valid_trades)) * 100 if len(valid_trades) > 0 else 0
        risultati_future.append({
            'Take_Profit': TP,
            'Stop_Loss': SL,
            'Net_PnL': round(pnl_totale, 2),
            'Win_Rate_%': round(win_rate, 2),
            'Wins': win_count,
            'Losses': loss_count,
            'Time_Stops': time_stop_count
        })

# --- 3. OUTPUT RISULTATI ---
df_future_res = pd.DataFrame(risultati_future).sort_values(by='Net_PnL', ascending=False).reset_index(drop=True)

print(f"\nOttimizzazione completata in {(time.time() - start_opt):.2f} secondi.")
print("Migliori 10 Configurazioni di Rischio sul FUTURE (Spread Rimosso, Commissioni Incluse):")

try:
    from IPython.display import display
    display(df_future_res.head(10))
except ImportError:
    print(df_future_res.head(10).to_string())

Avvio Backtest su FUTURE puro (Isolamento dell'Alpha)...

Ottimizzazione completata in 5.91 secondi.
Migliori 10 Configurazioni di Rischio sul FUTURE (Spread Rimosso, Commissioni Incluse):


,Take_Profit,Stop_Loss,Net_PnL,Win_Rate_%,Wins,Losses,Time_Stops
0,0.50,1.50,-21.10,63.45,486,82,198
1,0.50,1.25,-24.10,62.79,481,111,174
2,0.50,1.00,-35.35,61.36,470,167,129
3,0.50,0.75,-48.60,57.05,437,242,87
4,0.75,1.50,-61.35,42.95,329,110,327
5,0.75,1.25,-65.60,42.17,323,148,295
6,0.75,1.00,-74.35,41.12,315,214,237
7,1.00,1.50,-82.85,28.46,218,121,427
8,0.75,0.75,-83.60,37.73,289,303,174
9,0.50,0.50,-84.35,46.48,356,371,39


In [7]:
import pandas as pd
import numpy as np

# --- CONTROLLO DI SICUREZZA MEMORIA (JUPYTER) ---
# Verifica se i dati sono presenti in memoria, altrimenti avvisa l'utente
if 'valid_trades' not in globals() or 'df' not in globals():
    raise NameError(
        "\n\n🚨 ERRORE DI MEMORIA: Variabili 'valid_trades' o 'df' non trovate!\n"
        "Questo accade perché stai eseguendo questo script in un kernel vuoto.\n"
        "I file Jupyter Notebook non condividono la RAM tra loro.\n\n"
        "👉 SOLUZIONE: Copia l'intero contenuto di questa cella e incollalo in \n"
        "una nuova cella IN FONDO al notebook precedente (quello dove hai \n"
        "già calcolato i segnali e caricato i dati), dopodiché eseguilo lì.\n"
    )

print("Avvio Diagnostica: Analisi dei Movimenti a 30 secondi...")

movimenti_up = []
movimenti_down = []
moss_oltre_soglia = 0
soglia_test = 0.50 # Punti

for trade in valid_trades:
    t_ingresso = trade['Datetime_UTC']
    t_scadenza = t_ingresso + pd.Timedelta(seconds=30)
    prezzo_ingresso = trade['Price']
    
    # Isola i 30 secondi successivi
    finestra_futura = df.loc[t_ingresso : t_scadenza]
    if finestra_futura.empty:
        continue
        
    # Calcola il massimo movimento verso l'alto e verso il basso nei 30s
    max_price = finestra_futura['Price'].max()
    min_price = finestra_futura['Price'].min()
    
    move_up = max_price - prezzo_ingresso
    move_down = prezzo_ingresso - min_price
    
    movimenti_up.append(move_up)
    movimenti_down.append(move_down)
    
    # Verifica se si è mosso di oltre 0.50 pt in ALMENO una direzione
    if move_up >= soglia_test or move_down >= soglia_test:
        moss_oltre_soglia += 1

tot_trades = len(movimenti_up)
perc_movimento = (moss_oltre_soglia / tot_trades) * 100 if tot_trades > 0 else 0

print("\n--- RISULTATO DIAGNOSTICA (RICHIESTA DA CLAUDE) ---")
print(f"Trade analizzati: {tot_trades}")
print(f"Movimento medio Massimo UP: {np.mean(movimenti_up):.2f} punti")
print(f"Movimento medio Massimo DOWN: {np.mean(movimenti_down):.2f} punti")
print(f"Volatilità Massima Media: {max(np.mean(movimenti_up), np.mean(movimenti_down)):.2f} punti")
print(f"Il prezzo si è mosso di > {soglia_test} pt nel {perc_movimento:.2f}% dei casi entro 30 secondi.")

if perc_movimento < 60:
    print("-> CONCLUSIONE: Segnale morto (Ipotesi A/C confermata). Il mercato non si muove.")
else:
    print("-> CONCLUSIONE: Il mercato si muove eccome! Procediamo al test invertito (Ipotesi B).")

Avvio Diagnostica: Analisi dei Movimenti a 30 secondi...

--- RISULTATO DIAGNOSTICA (RICHIESTA DA CLAUDE) ---
Trade analizzati: 766
Movimento medio Massimo UP: 1.21 punti
Movimento medio Massimo DOWN: 1.31 punti
Volatilità Massima Media: 1.31 punti
Il prezzo si è mosso di > 0.5 pt nel 99.48% dei casi entro 30 secondi.
-> CONCLUSIONE: Il mercato si muove eccome! Procediamo al test invertito (Ipotesi B).


In [8]:
import pandas as pd
import numpy as np
import time

# Controllo di sicurezza memoria
if 'valid_trades' not in globals() or 'df' not in globals():
    raise NameError("Errore: Variabili non trovate. Esegui questa cella nel notebook originale!")

print("Avvio Backtest INVERTITO su FUTURE (Test Ipotesi B di Claude)...")
start_opt = time.time()

tp_list = [0.50, 0.75, 1.00, 1.25, 1.50]
sl_list = [0.50, 0.75, 1.00, 1.25, 1.50]
TIME_STOP_SEC = 10
COMMISSIONI_PT = 0.10 

risultati_invertiti = []

for TP in tp_list:
    for SL in sl_list:
        pnl_totale = 0
        win_count = 0
        loss_count = 0
        time_stop_count = 0
        
        for trade in valid_trades:
            t_ingresso = trade['Datetime_UTC']
            t_scadenza = t_ingresso + pd.Timedelta(seconds=TIME_STOP_SEC)
            
            # ---------------------------------------------------------
            # IL CAMBIAMENTO MAGICO: INVERTIAMO IL SEGNALE (* -1)
            # Se era 1 (Long da Reversal), ora diventa -1 (Short da Momentum)
            # Se era -1 (Short da Reversal), ora diventa 1 (Long da Momentum)
            # ---------------------------------------------------------
            direzione = trade['Signal'] * -1 
            
            finestra_futura = df.loc[t_ingresso : t_scadenza]
            if finestra_futura.empty:
                continue
                
            prezzo_ingresso = trade['Price'] # Esecuzione pura su Future
                
            if direzione == 1: # LONG
                target_price = prezzo_ingresso + TP
                stop_price = prezzo_ingresso - SL
                
                hit_tp = (finestra_futura['Price'] >= target_price).idxmax() if (finestra_futura['Price'] >= target_price).any() else None
                hit_sl = (finestra_futura['Price'] <= stop_price).idxmax() if (finestra_futura['Price'] <= stop_price).any() else None
                
                if hit_tp and hit_sl:
                    if hit_tp < hit_sl:
                        pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                    else:
                        pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                elif hit_tp:
                    pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                elif hit_sl:
                    pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                else:
                    prezzo_uscita = finestra_futura['Price'].iloc[-1]
                    pnl_totale += (prezzo_uscita - prezzo_ingresso) - COMMISSIONI_PT
                    time_stop_count += 1
                    
            elif direzione == -1: # SHORT
                target_price = prezzo_ingresso - TP
                stop_price = prezzo_ingresso + SL
                
                hit_tp = (finestra_futura['Price'] <= target_price).idxmax() if (finestra_futura['Price'] <= target_price).any() else None
                hit_sl = (finestra_futura['Price'] >= stop_price).idxmax() if (finestra_futura['Price'] >= stop_price).any() else None
                
                if hit_tp and hit_sl:
                    if hit_tp < hit_sl:
                        pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                    else:
                        pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                elif hit_tp:
                    pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                elif hit_sl:
                    pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                else:
                    prezzo_uscita = finestra_futura['Price'].iloc[-1]
                    pnl_totale += (prezzo_ingresso - prezzo_uscita) - COMMISSIONI_PT
                    time_stop_count += 1
        
        win_rate = (win_count / len(valid_trades)) * 100 if len(valid_trades) > 0 else 0
        risultati_invertiti.append({
            'Take_Profit': TP,
            'Stop_Loss': SL,
            'Net_PnL': round(pnl_totale, 2),
            'Win_Rate_%': round(win_rate, 2),
            'Wins': win_count,
            'Losses': loss_count,
            'Time_Stops': time_stop_count
        })

df_invertiti = pd.DataFrame(risultati_invertiti).sort_values(by='Net_PnL', ascending=False).reset_index(drop=True)

print(f"\nOttimizzazione Invertita completata in {(time.time() - start_opt):.2f} secondi.")
print("Migliori 10 Configurazioni (SEGNALE INVERTITO - Isolamento Alpha su Future):")
display(df_invertiti.head(10))

Avvio Backtest INVERTITO su FUTURE (Test Ipotesi B di Claude)...

Ottimizzazione Invertita completata in 5.64 secondi.
Migliori 10 Configurazioni (SEGNALE INVERTITO - Isolamento Alpha su Future):


,Take_Profit,Stop_Loss,Net_PnL,Win_Rate_%,Wins,Losses,Time_Stops
0,0.50,1.50,-6.10,62.92,482,56,228
1,0.50,1.25,-8.85,62.66,480,92,194
2,0.75,1.25,-31.35,44.26,339,121,306
3,0.50,1.00,-33.85,60.18,461,153,152
4,0.75,1.50,-34.10,44.39,340,80,346
5,1.00,1.25,-37.35,31.85,244,135,387
6,1.00,1.50,-40.10,31.98,245,90,431
7,1.25,1.50,-52.10,22.45,172,94,500
8,1.25,1.25,-52.35,22.19,170,143,453
9,0.75,1.00,-53.60,42.82,328,191,247


In [10]:
import pandas as pd
import numpy as np

# Controllo di sicurezza memoria
if 'valid_trades' not in globals() or 'df' not in globals():
    raise NameError("Errore: Esegui questa cella nel notebook originale che contiene i dati!")

print("Avvio Analisi Oraria (Win Rate per Fasce di Wall Street)...")

# 1. Convertiamo i valid_trades in un DataFrame
df_analisi = pd.DataFrame(valid_trades)

# 2. Soluzione Cross-Platform (Mac/Windows) per i Fusi Orari
# pd.to_datetime con utc=True garantisce che la colonna sia TZ-Aware senza errori
df_analisi['Datetime_UTC_Safe'] = pd.to_datetime(df_analisi['Datetime_UTC'], utc=True)
df_analisi['NY_Time'] = df_analisi['Datetime_UTC_Safe'].dt.tz_convert('America/New_York')
df_analisi['Ora_NY'] = df_analisi['NY_Time'].dt.hour

# 3. Simuliamo il trade Invertito (Momentum) con una config media: TP 0.75, SL 1.00
TP = 0.75
SL = 1.00
COMMISSIONI = 0.10

esiti = []
pnls = []

# Iteriamo sui trade (usiamo iterrows, veloce per poche centinaia di righe)
for _, trade in df_analisi.iterrows():
    t_ingresso = trade['Datetime_UTC']
    t_scadenza = t_ingresso + pd.Timedelta(seconds=10)
    direzione = trade['Signal'] * -1 # INVERTITO
    
    finestra_futura = df.loc[t_ingresso : t_scadenza]
    if finestra_futura.empty:
        esiti.append('SKIPPED'); pnls.append(0)
        continue
        
    prezzo_ingresso = trade['Price']
    pnl = 0
    esito = 'TIME_STOP'
    
    if direzione == 1: # LONG
        target = prezzo_ingresso + TP
        stop = prezzo_ingresso - SL
        hit_tp = (finestra_futura['Price'] >= target).idxmax() if (finestra_futura['Price'] >= target).any() else None
        hit_sl = (finestra_futura['Price'] <= stop).idxmax() if (finestra_futura['Price'] <= stop).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl: esito = 'WIN'; pnl = TP - COMMISSIONI
            else: esito = 'LOSS'; pnl = -SL - COMMISSIONI
        elif hit_tp: esito = 'WIN'; pnl = TP - COMMISSIONI
        elif hit_sl: esito = 'LOSS'; pnl = -SL - COMMISSIONI
        else: pnl = (finestra_futura['Price'].iloc[-1] - prezzo_ingresso) - COMMISSIONI
                
    elif direzione == -1: # SHORT
        target = prezzo_ingresso - TP
        stop = prezzo_ingresso + SL
        hit_tp = (finestra_futura['Price'] <= target).idxmax() if (finestra_futura['Price'] <= target).any() else None
        hit_sl = (finestra_futura['Price'] >= stop).idxmax() if (finestra_futura['Price'] >= stop).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl: esito = 'WIN'; pnl = TP - COMMISSIONI
            else: esito = 'LOSS'; pnl = -SL - COMMISSIONI
        elif hit_tp: esito = 'WIN'; pnl = TP - COMMISSIONI
        elif hit_sl: esito = 'LOSS'; pnl = -SL - COMMISSIONI
        else: pnl = (prezzo_ingresso - finestra_futura['Price'].iloc[-1]) - COMMISSIONI
            
    esiti.append(esito)
    pnls.append(pnl)

df_analisi['Esito'] = esiti
df_analisi['PnL'] = pnls

# 4. Raggruppiamo i risultati per Ora di New York
report_orario = df_analisi.groupby('Ora_NY').agg(
    Tot_Trades=('Esito', 'count'),
    Wins=('Esito', lambda x: (x == 'WIN').sum()),
    Losses=('Esito', lambda x: (x == 'LOSS').sum()),
    Time_Stops=('Esito', lambda x: (x == 'TIME_STOP').sum()),
    Net_PnL=('PnL', 'sum')
)

report_orario['Win_Rate_%'] = (report_orario['Wins'] / report_orario['Tot_Trades'] * 100).round(2)
report_orario['Net_PnL'] = report_orario['Net_PnL'].round(2)

print("\n--- REPORT ORARIO (Fuso Orario di Wall Street) ---")
print("I numeri indicano l'ora di NY (es. 9 = dalle 09:00 alle 09:59)")
display(report_orario)

Avvio Analisi Oraria (Win Rate per Fasce di Wall Street)...

--- REPORT ORARIO (Fuso Orario di Wall Street) ---
I numeri indicano l'ora di NY (es. 9 = dalle 09:00 alle 09:59)


,Tot_Trades,Wins,Losses,Time_Stops,Net_PnL,Win_Rate_%
Ora_NY,,,,,,
0,4,2,1,1,-0.65,50.00
1,2,1,0,1,0.05,50.00
2,3,0,1,2,-1.80,0.00
3,4,2,1,1,-0.15,50.00
6,2,2,0,0,1.30,100.00
7,5,2,1,2,1.00,40.00
8,3,0,0,3,-1.30,0.00
9,6,3,3,0,-1.35,50.00
10,10,4,1,5,0.00,40.00
